# Bead-pull calibration

Map the **stepper-motor step count** to the **bead position** for every pass of
the thread through the booster (each pass = one *sub-thread*).

For each sub-thread you record, at both ends, two things:

* the **motor step count** &mdash; **start** (the bead-position zero of that
  sub-thread) and **end** (the far end of the pass), and
* the **(x, y, z) position in the booster frame** &mdash; `start_xyz_m` and
  `end_xyz_m`.

From those the code derives everything else per sub-thread: the travel
*direction* and step *length* from the step anchors, the physical *length* from
the distance between the two 3-D endpoints, and the **step&harr;metre
conversion** from the two together (step span &divide; physical length).  There
is **no single global `steps_per_meter`** any more &mdash; each sub-thread carries
its own, so nothing extra has to be measured or entered.

> Because the 3-D endpoints now set the scale, their coordinates must be
> accurate: the distance between `start_xyz_m` and `end_xyz_m` *is* the physical
> length the step span is divided by.

Positions are taken relative to the homing limit switch (`gotoswitch`), so this
calibration stays valid across power cycles.  Stepper settings come from the
shared `config/measurement_config.json`.

This notebook produces the **calibration file**: per sub-thread its `index`,
`name`, the non-measurement margins, the **(x, y, z) start/end points**, and the
measured `start_steps` / `end_steps`.

**Workflow:** run the cells top-to-bottom.  Section 5 is the loop you repeat for
each sub-thread: jog the bead to the start, capture it; jog to the end, capture
it.

## 1. Imports and configuration

In [ ]:
import json
import sys
from pathlib import Path

# Make ``src`` importable (this notebook lives at the repo root).
sys.path.insert(0, str(Path.cwd()))

from src.bead_pull_controller import (
    Calibration,
    SubThreadCalibration,
    Stepper,
    SimulatedMotor,
)

# ---- edit these -----------------------------------------------------------
SIMULATE = True          # True: no hardware (dry test of the flow). False: real motor.

# The passes of the thread through the booster you will calibrate, in order.
# For each pass set:
#   index          : integer id
#   name           : label (shown in logs)
#   margin_start_m : skip this many metres after the start (no measurement there)
#   margin_end_m   : skip this many metres before the end (no measurement there)
#   start_xyz_m    : (x, y, z) of the START point in the booster frame, metres
#   end_xyz_m      : (x, y, z) of the END point in the booster frame, metres
# start_xyz_m / end_xyz_m are REQUIRED: the distance between them is the physical
# length that fixes this sub-thread's step<->metre scale, so enter them
# accurately.  The step anchors (start_steps / end_steps) are captured below by
# jogging.
SUB_THREADS = [
    {"index": 0, "name": "z0", "margin_start_m": 0.0, "margin_end_m": 0.0,
     "start_xyz_m": [-0.10, 0.0, 0.000], "end_xyz_m": [0.10, 0.0, 0.000]},
    {"index": 1, "name": "z1", "margin_start_m": 0.0, "margin_end_m": 0.0,
     "start_xyz_m": [-0.10, 0.0, 0.020], "end_xyz_m": [0.10, 0.0, 0.020]},
    {"index": 2, "name": "z2", "margin_start_m": 0.0, "margin_end_m": 0.0,
     "start_xyz_m": [-0.10, 0.0, 0.040], "end_xyz_m": [0.10, 0.0, 0.040]},
    {"index": 3, "name": "z3", "margin_start_m": 0.0, "margin_end_m": 0.0,
     "start_xyz_m": [-0.10, 0.0, 0.060], "end_xyz_m": [0.10, 0.0, 0.060]},
]

# ---- from the shared measurement config -----------------------------------
# Stepper settings and the calibration-file path live here.
MEASUREMENT_CONFIG = Path("config/measurement_config.json")
_cfg = json.loads(MEASUREMENT_CONFIG.read_text(encoding="utf-8"))
STEPPER = _cfg.get("stepper", {})
CALIBRATION_PATH = Path(_cfg.get("calibration_file", "config/bead_pull_calibration.json"))

print(f"Stepper: port={STEPPER.get('port')}, baud={STEPPER.get('baud')}, motor={STEPPER.get('motor')}")
print(f"sub-threads to calibrate: {[d['index'] for d in SUB_THREADS]}")
print(f"Calibration will be saved to: {CALIBRATION_PATH}")

## 2. Connect to the motor

In [ ]:
if SIMULATE:
    motor = SimulatedMotor(verbose=True)
    print("SIMULATE = True: using an in-memory motor (no serial port opened).")
else:
    motor = Stepper.open(**STEPPER)   # the config keys match Stepper.open's arguments
    print("Connected to", STEPPER.get("port"))

## 3. Home to the limit switch

This drives the motor to the limit switch and zeroes the step counter, so every
position you capture below is measured from the same physical reference.

In [ ]:
motor.home()
print("Homed. Position now:", motor.get_position(), "steps")


## 4. Jog helpers

`jog(steps)` moves the bead by a relative number of steps (positive vs. negative
= the two directions; start with small values and watch which way the bead goes)
and prints the new absolute position.  `pos()` just prints the current position.
These helpers are used to capture each sub-thread's endpoints (section 5).

In [ ]:
def jog(steps):
    """Move the bead by a relative number of steps and report the new position."""
    motor.move_by(int(steps))
    p = motor.get_position()
    print(f"jogged {int(steps):+d} -> {p} steps")
    return p

def pos():
    p = motor.get_position()
    print("position:", p, "steps")
    return p

print("Helpers ready: jog(steps), pos().")

### Jog cell — run/edit this as many times as you need

In [ ]:
# Examples (edit the number, re-run):
jog(500)
# jog(-200)
# pos()


## 5. Capture each sub-thread's start / end

Using `jog(...)` / `pos()` from section 4, for **each** sub-thread:

1. `jog(...)` until the bead sits at the sub-thread's **start** (zero), then run
   `capture(index, "start")`.
2. `jog(...)` until the bead sits at the sub-thread's **end**, then run
   `capture(index, "end")`.

You can revisit any sub-thread and capture again to overwrite a bad value.

In [ ]:
captured = {}   # index -> {"start_steps": ..., "end_steps": ...}

def capture(index, which):
    """Record the current position as the start/end of a sub-thread."""
    which = which.lower()
    assert which in ("start", "end"), "which must be 'start' or 'end'"
    p = motor.get_position()
    captured.setdefault(index, {})[f"{which}_steps"] = p
    print(f"sub-thread {index}: {which} = {p} steps")
    return p

print("Helper ready: capture(index, 'start'|'end').  (jog/pos come from section 4.)")

### Capture cells — run when the bead is at the start / end of a sub-thread

In [ ]:
# When the bead is at the START (zero) of, say, sub-thread 0:
capture(0, "start")


In [ ]:
# When the bead is at the END of sub-thread 0:
capture(0, "end")


Repeat the jog + capture cells for every entry in `SUB_THREADS` (indices
`1`, `2`, ... here).  You can copy the two capture lines and change the index, or
just re-run them after editing the index.  Check progress any time:

In [ ]:
for d in SUB_THREADS:
    i = d["index"]
    got = captured.get(i, {})
    print(f"sub-thread {i} ({d['name']}): "
          f"start={got.get('start_steps', '---')}, end={got.get('end_steps', '---')}")


> **Tip for a dry test (`SIMULATE = True`):** there is no real bead, so just move
> the simulated motor before capturing, e.g. `motor.move_by(0); capture(0,'start')`
> then `motor.move_by(20000); capture(0,'end')`, to exercise the save below.

## 6. Build and save the calibration

In [ ]:
sub_threads = []
for d in SUB_THREADS:
    i = d["index"]
    if i not in captured or "start_steps" not in captured[i] or "end_steps" not in captured[i]:
        raise ValueError(f"sub-thread {i} ({d.get('name')}) is missing a start or end capture.")
    if d.get("start_xyz_m") is None or d.get("end_xyz_m") is None:
        raise ValueError(f"sub-thread {i} ({d.get('name')}) is missing start_xyz_m / end_xyz_m.")
    sub_threads.append(
        SubThreadCalibration(
            index=i,
            name=d.get("name"),
            margin_start_m=d.get("margin_start_m", 0.0),
            margin_end_m=d.get("margin_end_m", 0.0),
            start_xyz_m=d["start_xyz_m"],
            end_xyz_m=d["end_xyz_m"],
            start_steps=captured[i]["start_steps"],
            end_steps=captured[i]["end_steps"],
        )
    )

# Each sub-thread's step<->metre scale is derived from its own anchors (step span
# / distance between the 3-D endpoints); the summary's steps/m column shows it.
calibration = Calibration(sub_threads=sub_threads)
print(calibration.summary())
path = calibration.save(CALIBRATION_PATH)
print("\nSaved calibration to:", path)

## 7. Verify (reload from disk)

In [ ]:
# Reload the saved calibration (anchors + name + 3-D endpoints + margins),
# exactly as run_bead_pull_measurement.py does at load time.  The step<->metre
# scale is derived per sub-thread from the anchors, so nothing else is passed.
reloaded = Calibration.from_calibration_file(CALIBRATION_PATH)
print(reloaded.summary())

## 8. Release the motor

Switches the coils off and closes the serial port (no-op for the simulated
motor).

In [ ]:
motor.shutdown()
print("Motor released.")
